In [1]:
import sys
sys.path.append("..")
import os
import torch
import json
import yaml
from pathlib import Path
from datasets import DatasetDict, load_from_disk
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

from transformers import AutoTokenizer, AutoModelForCausalLM
from src.data import PROJECT_ROOT
from src.dataset_cot import prepare_direct_pythia_dataset, get_cot_prompt_template, evaluate_cot_capability, evaluate_direct_pythia
from src.llm_upgrade import (
    prepare_model_for_finetune,
    train_lora_model,
    save_finetuned_model,
    wrap_for_transformer_lens,
    create_quantization_config
)

# Параметры (q)LoRA

In [2]:
EXP_ID = "exp4"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
VARIANT = "depth-1"
USE_SMALL = False

In [3]:
EPOCHS = 4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1.5e-4
MAX_LENGTH = 512
SAVE_STEPS = 100
LOGGING_STEPS = 20

In [4]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

In [5]:
TARGET_MODULES = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

In [6]:
OUTPUT_DIR = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}")
ADAPTER_SAVE_PATH = str(PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/lora_final")

In [7]:
cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}"
raw_dataset = load_from_disk(str(cache_path))

In [8]:
TRAIN_SIZE = 10000
DEV_SIZE = 1000
SEED = 42

# Подготовка датасета

In [9]:
train_subset = raw_dataset["train"].shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(raw_dataset["train"]))))
dev_subset = raw_dataset["dev"].shuffle(seed=SEED).select(range(min(DEV_SIZE, len(raw_dataset["dev"]))))

In [10]:
train_dataset = prepare_direct_pythia_dataset(train_subset)
dev_dataset   = prepare_direct_pythia_dataset(dev_subset)

In [11]:
len(train_dataset)

10000

In [12]:
len(dev_subset)

1000

In [13]:
direct_dir = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_direct_pythia"
if dev_dataset:
    DatasetDict({"train": train_dataset, "dev": dev_dataset, "test": raw_dataset["test"]}).save_to_disk(str(direct_dir))

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/17788 [00:00<?, ? examples/s]

# Обучение

In [14]:
# qlora
quantization_config = create_quantization_config(use_4bit=True)

In [15]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"  # Flash Attention часто нестабилен с LoRA на Windows
)

`torch_dtype` is deprecated! Use `dtype` instead!


In [17]:
base_model = prepare_model_for_kbit_training(base_model)

In [18]:
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False
)

In [19]:
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

trainable params: 8,388,608 || all params: 1,020,170,240 || trainable%: 0.8223


In [20]:
train_config = {
    "output_dir": OUTPUT_DIR,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "logging_steps": LOGGING_STEPS,
    "save_steps": SAVE_STEPS,
    "use_wandb": False
}

In [21]:
trained_model, metrics = train_lora_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    config=train_config,
    max_length=MAX_LENGTH
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,1.446900,1.000515
100,0.825900,0.826776
150,0.781800,0.788090
200,0.742400,0.773095
250,0.735800,0.763379
300,0.748000,0.758818
350,0.727000,0.745421
400,0.718400,0.745811
450,0.724000,0.740649
500,0.697100,0.731850


c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\mephi_diss\.venv\Lib\site-packages\torch\_dynamo\eval_frame.py:838: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\MyPythonProjects\meph

In [22]:
save_finetuned_model(trained_model, tokenizer, ADAPTER_SAVE_PATH)

Адаптер сохранён в C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp4\lora_final


# Тест дообученной модели

In [14]:
BEST_CHECKPOINT = 1700
adapter_path = PROJECT_ROOT / f"results/checkpoints/finetune/{EXP_ID}/checkpoint-{BEST_CHECKPOINT}"

In [15]:
hooked_model, _ = wrap_for_transformer_lens(
    MODEL_NAME,
    str(adapter_path),
    device="cuda"
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model EleutherAI/pythia-1b-deduped into HookedTransformer


In [16]:
eval_prefix = "{theory} {assertion} The assertion is"

In [17]:
full_test = load_from_disk(str(cache_path))["test"]

In [27]:
test_acc = evaluate_cot_capability(
    model=hooked_model,
    dataset=full_test,
    prompt_template=eval_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating CoT capability: 100%|██████████| 556/556 [14:59<00:00,  1.62s/it]


In [ ]:
test_acc2 = evaluate_direct_pythia(
    model=hooked_model,
    dataset=full_test,
    prompt_template=eval_prefix,
    batch_size=32,
    device="cuda"
)

Evaluating direct Pythia: 100%|██████████| 556/556 [11:33<00:00,  1.25s/it]


In [28]:
results = {
    "experiment_id": EXP_ID,
    "model": MODEL_NAME,
    "variant": VARIANT,
    "best_checkpoint": adapter_path.name,
    "val_loss_at_checkpoint": 0.715634,
    "test_accuracy": test_acc,
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "train_size": TRAIN_SIZE
    }
}

In [29]:
results_path = Path(OUTPUT_DIR) / "results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

In [30]:
print(f"Experiment: {EXP_ID}")
print(f"Checkpoint: {adapter_path.name}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Results saved: {results_path}")

Experiment: exp4
Checkpoint: checkpoint-1700
Test Accuracy: 0.5302
Results saved: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp4\results.json


In [ ]:
results2 = {
    "experiment_id": EXP_ID,
    "model": MODEL_NAME,
    "variant": VARIANT,
    "best_checkpoint": adapter_path.name,
    "val_loss_at_checkpoint": 0.715634,
    "test_accuracy": test_acc, # исправить на test_acc2
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lora_r": LORA_R, "train_size": TRAIN_SIZE
    }
}

In [22]:
results_path2 = Path(OUTPUT_DIR) / "results2.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results2, f, indent=2, ensure_ascii=False)

In [23]:
print(f"Experiment: {EXP_ID}")
print(f"Checkpoint: {adapter_path.name}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Results saved: {results_path2}")

Experiment: exp4
Checkpoint: checkpoint-1700
Test Accuracy: 0.5489
Results saved: C:\MyPythonProjects\mephi_diss\results\checkpoints\finetune\exp4\results2.json


# Диагностика предсказаний

In [24]:
test_examples = raw_dataset["test"].select(range(5))
eval_prefix = "{theory} {assertion} The assertion is"

In [25]:
with torch.no_grad():
    for i, ex in enumerate(test_examples):
        prompt = eval_prefix.format(theory=ex["theory"], assertion=ex["assertion"])
        tokens = hooked_model.to_tokens([prompt], prepend_bos=True).to("cuda")
        logits = hooked_model(tokens)

        # Индексы целевых токенов
        true_id = hooked_model.tokenizer.encode("True", add_special_tokens=False)[-1]
        false_id = hooked_model.tokenizer.encode("False", add_special_tokens=False)[-1]

        # Длина промпта (последний токен находится по этому индексу после добавления BOS)
        prompt_len = len(hooked_model.tokenizer.encode(prompt, add_special_tokens=False))

        logit_t = logits[0, prompt_len, true_id].item()
        logit_f = logits[0, prompt_len, false_id].item()
        pred = "True" if logit_t > logit_f else "False"
        true_label = "True" if ex["label"] else "False"

        print(f"[{i+1}] Предсказание: {pred:5s} | Логиты T/F: {logit_t:+.2f} / {logit_f:+.2f} | Истина: {true_label} | Совпало: {pred == true_label}")

[1] Предсказание: True  | Логиты T/F: +15.50 / +15.31 | Истина: True | Совпало: True
[2] Предсказание: False | Логиты T/F: +15.00 / +15.38 | Истина: False | Совпало: True
[3] Предсказание: True  | Логиты T/F: +15.44 / +15.25 | Истина: True | Совпало: True
[4] Предсказание: False | Логиты T/F: +15.12 / +15.62 | Истина: False | Совпало: True
[5] Предсказание: False | Логиты T/F: +14.94 / +15.38 | Истина: True | Совпало: False


In [26]:
del hooked_model
torch.cuda.empty_cache()